# 14 — 3D overlays

This notebook is for visual comparison:

- RGB overlays
- layer-specific RGB sliders
- total-count grayscale background + colored element overlays

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re

import pandas as pd
import plotly.io as pio

from pymagsims import SIMSVolume
from pymagsims.plotting import (
    plot_rgb_overlay,
    plot_rgb_overlay_slider,
    plot_element_overlay_slider,
)

pio.renderers.default = "iframe"

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"
IMAGE_SHAPE = (256, 256)
CHANNEL_BIN_FILE = DATA / "bins" / "channel_bins_from_3d_csv_calibration.csv"

## 1. Rebuild volume and element volumes

In [ ]:
def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(RAW_LAYER_DIR.glob("*Image_*.raw"), key=natural_sort_key)
selected_bins = pd.read_csv(CHANNEL_BIN_FILE)

volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=selected_bins,
    spectrum=None,
    include_total=True,
    shape=IMAGE_SHAPE,
)

elements = ["Ti", "F", "Mg"]

element_volumes = {}

for element in elements:
    try:
        element_volumes[element] = volume.summed_by_element(selected_bins, element)
        print(element, element_volumes[element].sum())
    except ValueError as exc:
        print(f"Skipping {element}: {exc}")

element_volumes.keys()

## 2. RGB overlay from summed projections

In [ ]:
# Pick three channels. Change these names to your elements.
red_label, green_label, blue_label = list(element_volumes.keys())[:3]

plot_rgb_overlay(
    red=element_volumes[red_label].sum(axis=0),
    green=element_volumes[green_label].sum(axis=0),
    blue=element_volumes[blue_label].sum(axis=0),
    red_label=red_label,
    green_label=green_label,
    blue_label=blue_label,
    log=True,
)

## 3. RGB overlay slider through layers

In [ ]:
plot_rgb_overlay_slider(
    red=element_volumes[red_label],
    green=element_volumes[green_label],
    blue=element_volumes[blue_label],
    red_label=red_label,
    green_label=green_label,
    blue_label=blue_label,
    log=True,
).show()

## 4. Total counts as gray background with colored element overlays

In [ ]:
plot_element_overlay_slider(
    background=volume.get("Total"),
    overlays={
        label: arr for label, arr in element_volumes.items()
    },
    colors={
        "Ti": (1.0, 0.2, 0.0),
        "Si": (0.0, 0.7, 1.0),
        "Mg": (1.0, 0.0, 1.0),
        "Hf": (1.0, 0.8, 0.0),
        "Au": (1.0, 0.75, 0.0),
    },
    alpha=0.75,
    log=True,
    background_scale=0.75,
    background_gamma=1.2,
    overlay_gamma=0.5,
).show()

## Tuning guide

If overlays are too weak:

```python
alpha=0.9
overlay_gamma=0.4
```

If the background is too weak:

```python
background_scale=0.8
background_gamma=1.0
```

If the background washes out overlays:

```python
background_scale=0.5
background_gamma=1.5
```